In [1]:
# ============================================================
# PHASE 5 — CELL 1
# Imports
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import traceback
import gc

import cv2
import numpy as np
import pandas as pd

from tqdm.auto import tqdm


print("=" * 70)
print("PHASE 5 — MOUTH ROI EXTRACTION")
print("=" * 70)

print("OpenCV :", cv2.__version__)
print("NumPy  :", np.__version__)
print("Pandas :", pd.__version__)

PHASE 5 — MOUTH ROI EXTRACTION
OpenCV : 4.11.0
NumPy  : 1.26.4
Pandas : 2.3.1


In [2]:
# ============================================================
# PHASE 5 — CELL 2
# Production Configuration
# ============================================================

from pathlib import Path


# ============================================================
# Project
# ============================================================

PROJECT_DIR = Path.cwd().resolve()

while PROJECT_DIR != PROJECT_DIR.parent:

    if (
        (PROJECT_DIR / "output").exists()
        or
        (PROJECT_DIR / "dataset").exists()
    ):
        break

    PROJECT_DIR = PROJECT_DIR.parent


OUTPUT_DIR = (
    PROJECT_DIR /
    "output"
)


# ============================================================
# Input / Output Folder Names
# ============================================================

ALIGNED_FACE_DIR_NAME = (
    "aligned_face"
)

LANDMARK_DIR_NAME = (
    "landmarks_aligned"
)

ALIGNMENT_METADATA_NAME = (
    "alignment_metadata.csv"
)

MOUTH_DIR_NAME = (
    "mouth_crop"
)

MOUTH_PREVIEW_DIR_NAME = (
    "mouth_preview"
)

MOUTH_METADATA_NAME = (
    "mouth_metadata.csv"
)

MOUTH_SUMMARY_NAME = (
    "mouth_summary.csv"
)

MOUTH_STATE_NAME = (
    "phase5_state.json"
)

MOUTH_LOG_NAME = (
    "phase5_log.txt"
)


# ============================================================
# Mouth Crop
# ============================================================

MOUTH_SIZE = 96

MOUTH_PADDING = 0.35

MIN_MOUTH_SIZE = 8

MAX_MOUTH_SIZE_RATIO = 0.90


# ============================================================
# Preview
# ============================================================

SAVE_PREVIEW = True

PREVIEW_MAX_IMAGES = 100


# ============================================================
# Processing
# ============================================================

FORCE_RERUN = False

RESUME_ENABLED = True


# ============================================================
# Quality
# ============================================================

MIN_BLUR_SCORE = 20.0

MIN_BRIGHTNESS = 15.0

MAX_BRIGHTNESS = 245.0


# ============================================================
# Supported image extensions
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
}


print()
print("=" * 70)
print("Phase 5 Configuration")
print("=" * 70)

print("Project :", PROJECT_DIR)
print("Output  :", OUTPUT_DIR)
print("Mouth size :", MOUTH_SIZE)
print("Padding   :", MOUTH_PADDING)
print("Resume    :", RESUME_ENABLED)
print("Force run :", FORCE_RERUN)
print("=" * 70)


Phase 5 Configuration
Project : C:\LipReadingSSL
Output  : C:\LipReadingSSL\output
Mouth size : 96
Padding   : 0.35
Resume    : True
Force run : False


In [3]:
# ============================================================
# PHASE 5 — CELL 3
# MediaPipe Face Mesh Mouth Landmarks
# ============================================================

# MediaPipe Face Mesh mouth landmarks
#
# ใช้ landmark ชุดเดียวกับ MediaPipe Face Mesh
# ไม่ใช่ InsightFace 106 landmarks

MOUTH_LANDMARKS = [
    61, 146, 91, 181, 84,
    17, 314, 405, 321, 375,
    291, 308, 324, 318, 402,
    317, 14, 87, 178, 88,
    95, 185, 40, 39, 37,
    0, 267, 269, 270, 409,
    415, 310, 311, 312, 13,
    82, 81, 42, 183, 78,
]


print("=" * 70)
print("Cell 3 — MediaPipe Mouth Landmark Configuration")
print("=" * 70)

print(
    f"Mouth landmarks: "
    f"{len(MOUTH_LANDMARKS)}"
)

print(
    f"Minimum landmark index: "
    f"{min(MOUTH_LANDMARKS)}"
)

print(
    f"Maximum landmark index: "
    f"{max(MOUTH_LANDMARKS)}"
)

Cell 3 — MediaPipe Mouth Landmark Configuration
Mouth landmarks: 40
Minimum landmark index: 0
Maximum landmark index: 415


In [4]:
# ============================================================
# PHASE 5 — CELL 4
# Path Helpers
# ============================================================

def get_video_id(video_output_dir):

    return Path(
        video_output_dir
    ).resolve().name


def get_phase5_paths(video_output_dir):

    video_output_dir = (
        Path(video_output_dir)
        .resolve()
    )

    return {

        "video_output":
            video_output_dir,

        "aligned_face":
            video_output_dir /
            ALIGNED_FACE_DIR_NAME,

        "landmarks":
            video_output_dir /
            LANDMARK_DIR_NAME,

        "alignment_metadata":
            video_output_dir /
            ALIGNMENT_METADATA_NAME,

        "mouth_crop":
            video_output_dir /
            MOUTH_DIR_NAME,

        "mouth_preview":
            video_output_dir /
            MOUTH_PREVIEW_DIR_NAME,

        "mouth_metadata":
            video_output_dir /
            MOUTH_METADATA_NAME,

        "mouth_summary":
            video_output_dir /
            MOUTH_SUMMARY_NAME,

        "state":
            video_output_dir /
            MOUTH_STATE_NAME,

        "log":
            video_output_dir /
            MOUTH_LOG_NAME,
    }


def ensure_phase5_directories(
    video_output_dir
):

    paths = get_phase5_paths(
        video_output_dir
    )

    paths["mouth_crop"].mkdir(
        parents=True,
        exist_ok=True
    )

    if SAVE_PREVIEW:

        paths["mouth_preview"].mkdir(
            parents=True,
            exist_ok=True
        )

    return paths


print("=" * 70)
print("Cell 4 — Path Helpers Ready")
print("=" * 70)

Cell 4 — Path Helpers Ready


In [5]:
# ============================================================
# PHASE 5 — CELL 5
# Frame / Landmark File Matching
# ============================================================

def extract_frame_number(
    path
):

    stem = Path(path).stem

    digits = ""

    for char in reversed(stem):

        if char.isdigit():

            digits = (
                char +
                digits
            )

        else:

            if digits:
                break

    if not digits:

        return None

    return int(digits)


def discover_aligned_frames(
    aligned_dir
):

    aligned_dir = Path(
        aligned_dir
    )

    if not aligned_dir.exists():

        raise FileNotFoundError(
            f"aligned_face not found:\n"
            f"{aligned_dir}"
        )

    files = [

        p

        for p in aligned_dir.iterdir()

        if (
            p.is_file()
            and
            p.suffix.lower()
            in IMAGE_EXTENSIONS
        )
    ]

    files.sort(
        key=lambda p: (
            extract_frame_number(p)
            if extract_frame_number(p)
            is not None
            else 10**18,
            p.name
        )
    )

    return files


def build_landmark_map(
    landmark_dir
):

    landmark_dir = Path(
        landmark_dir
    )

    if not landmark_dir.exists():

        raise FileNotFoundError(
            f"landmarks_aligned not found:\n"
            f"{landmark_dir}"
        )

    landmark_files = [

        p

        for p in landmark_dir.iterdir()

        if (
            p.is_file()
            and
            p.suffix.lower()
            == ".npy"
        )
    ]

    result = {}

    for path in landmark_files:

        frame_number = (
            extract_frame_number(path)
        )

        if frame_number is None:
            continue

        result[
            frame_number
        ] = path

    return result


print("=" * 70)
print("Cell 5 — Frame Matching Ready")
print("=" * 70)

Cell 5 — Frame Matching Ready


In [6]:
# ============================================================
# PHASE 5 — CELL 6
# Landmark Validation
# ============================================================

def load_landmarks(
    landmark_path
):

    try:

        landmarks = np.load(
            landmark_path
        )

    except Exception:

        return None, "load_error"


    if not isinstance(
        landmarks,
        np.ndarray
    ):

        return None, "not_numpy_array"


    if landmarks.ndim != 2:

        return None, "invalid_dimensions"


    if landmarks.shape[0] < 468:

        return None, "insufficient_landmarks"


    if landmarks.shape[1] < 2:

        return None, "invalid_coordinate_dimension"


    landmarks = (
        landmarks[:, :2]
        .astype(np.float32)
    )


    if not np.isfinite(
        landmarks
    ).all():

        return None, "non_finite_landmarks"


    return landmarks, "ok"


def normalize_landmarks_to_pixels(
    landmarks,
    image_width,
    image_height
):

    landmarks = (
        landmarks
        .astype(np.float32)
        .copy()
    )

    max_x = float(
        np.max(
            landmarks[:, 0]
        )
    )

    max_y = float(
        np.max(
            landmarks[:, 1]
        )
    )

    min_x = float(
        np.min(
            landmarks[:, 0]
        )
    )

    min_y = float(
        np.min(
            landmarks[:, 1]
        )
    )


    # --------------------------------------------------------
    # Normalized MediaPipe coordinates
    # --------------------------------------------------------

    if (
        min_x >= -0.05
        and
        min_y >= -0.05
        and
        max_x <= 1.05
        and
        max_y <= 1.05
    ):

        landmarks[:, 0] *= image_width

        landmarks[:, 1] *= image_height


    return landmarks


print("=" * 70)
print("Cell 6 — Landmark Validation Ready")
print("=" * 70)

Cell 6 — Landmark Validation Ready


In [7]:
# ============================================================
# PHASE 5 — CELL 7
# Mouth ROI Calculation
# ============================================================

def calculate_mouth_roi(
    image,
    landmarks
):

    if image is None:

        return None, (
            "invalid_image"
        )


    height, width = (
        image.shape[:2]
    )


    landmarks = (
        normalize_landmarks_to_pixels(
            landmarks,
            width,
            height
        )
    )


    mouth = (
        landmarks[
            MOUTH_LANDMARKS
        ]
    )


    xmin = float(
        np.min(
            mouth[:, 0]
        )
    )

    xmax = float(
        np.max(
            mouth[:, 0]
        )
    )

    ymin = float(
        np.min(
            mouth[:, 1]
        )
    )

    ymax = float(
        np.max(
            mouth[:, 1]
        )
    )


    mouth_width = (
        xmax - xmin
    )

    mouth_height = (
        ymax - ymin
    )


    if (
        mouth_width
        < MIN_MOUTH_SIZE
        or
        mouth_height
        < MIN_MOUTH_SIZE
    ):

        return None, (
            "mouth_too_small"
        )


    # --------------------------------------------------------
    # Make square ROI
    # --------------------------------------------------------

    cx = (
        xmin + xmax
    ) / 2.0

    cy = (
        ymin + ymax
    ) / 2.0


    mouth_size = max(
        mouth_width,
        mouth_height
    )


    mouth_size *= (
        1.0 +
        MOUTH_PADDING
    )


    # --------------------------------------------------------
    # Prevent ROI from being too large
    # --------------------------------------------------------

    max_size = (
        min(width, height)
        *
        MAX_MOUTH_SIZE_RATIO
    )

    mouth_size = min(
        mouth_size,
        max_size
    )


    half = (
        mouth_size / 2.0
    )


    x1 = int(
        round(cx - half)
    )

    y1 = int(
        round(cy - half)
    )

    x2 = int(
        round(cx + half)
    )

    y2 = int(
        round(cy + half)
    )


    # --------------------------------------------------------
    # Clamp
    # --------------------------------------------------------

    x1 = max(
        0,
        x1
    )

    y1 = max(
        0,
        y1
    )

    x2 = min(
        width,
        x2
    )

    y2 = min(
        height,
        y2
    )


    if (
        x2 <= x1
        or
        y2 <= y1
    ):

        return None, (
            "invalid_crop"
        )


    crop = image[
        y1:y2,
        x1:x2
    ]


    if crop.size == 0:

        return None, (
            "empty_crop"
        )


    crop = cv2.resize(
        crop,
        (
            MOUTH_SIZE,
            MOUTH_SIZE
        ),
        interpolation=cv2.INTER_AREA
    )


    return {

        "crop":
            crop,

        "center_x":
            float(cx),

        "center_y":
            float(cy),

        "roi_size":
            float(mouth_size),

        "x1":
            int(x1),

        "y1":
            int(y1),

        "x2":
            int(x2),

        "y2":
            int(y2),

        "mouth_width":
            float(mouth_width),

        "mouth_height":
            float(mouth_height),

    }, "ok"


print("=" * 70)
print("Cell 7 — Mouth ROI Calculator Ready")
print("=" * 70)

Cell 7 — Mouth ROI Calculator Ready


In [8]:
# ============================================================
# PHASE 5 — CELL 8
# Mouth Crop Quality
# ============================================================

def calculate_blur_score(
    image
):

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    return float(
        cv2.Laplacian(
            gray,
            cv2.CV_64F
        ).var()
    )


def calculate_brightness(
    image
):

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    return float(
        np.mean(gray)
    )


def evaluate_crop_quality(
    crop
):

    if crop is None:

        return {
            "score": 0.0,
            "blur_score": 0.0,
            "brightness": 0.0,
            "status": "invalid_crop",
        }


    blur_score = (
        calculate_blur_score(
            crop
        )
    )

    brightness = (
        calculate_brightness(
            crop
        )
    )


    if (
        blur_score
        < MIN_BLUR_SCORE
    ):

        status = "low_sharpness"

    elif (
        brightness
        < MIN_BRIGHTNESS
    ):

        status = "too_dark"

    elif (
        brightness
        > MAX_BRIGHTNESS
    ):

        status = "too_bright"

    else:

        status = "good"


    score = (
        min(
            blur_score /
            MIN_BLUR_SCORE,
            2.0
        )
    )


    return {

        "score":
            float(score),

        "blur_score":
            float(blur_score),

        "brightness":
            float(brightness),

        "status":
            status,
    }


print("=" * 70)
print("Cell 8 — Quality System Ready")
print("=" * 70)

Cell 8 — Quality System Ready


In [9]:
# ============================================================
# PHASE 5 — CELL 9
# State Management
# ============================================================

def utc_now_iso():

    return (
        datetime
        .now(timezone.utc)
        .isoformat()
    )


def read_phase5_state(
    video_output_dir
):

    paths = get_phase5_paths(
        video_output_dir
    )

    state_path = (
        paths["state"]
    )

    if not state_path.exists():

        return {

            "status":
                "NOT_STARTED",

            "next_index":
                0,

            "updated_at":
                None,

        }


    try:

        with open(
            state_path,
            "r",
            encoding="utf-8"
        ) as f:

            return json.load(f)

    except Exception:

        return {

            "status":
                "STATE_READ_ERROR",

            "next_index":
                0,

            "updated_at":
                None,

        }


def write_phase5_state(
    video_output_dir,
    state
):

    paths = get_phase5_paths(
        video_output_dir
    )

    state_path = (
        paths["state"]
    )

    state = dict(state)

    state[
        "updated_at"
    ] = utc_now_iso()


    temp_path = (
        state_path.with_suffix(
            ".tmp"
        )
    )


    with open(
        temp_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            state,
            f,
            indent=2,
            ensure_ascii=False
        )


    try:

        temp_path.replace(
            state_path
        )

    except PermissionError:

        # Windows-safe fallback
        if state_path.exists():

            state_path.unlink()

        temp_path.replace(
            state_path
        )


print("=" * 70)
print("Cell 9 — State Management Ready")
print("=" * 70)

Cell 9 — State Management Ready


In [10]:
# ============================================================
# PHASE 5 — CELL 10
# Main Mouth Crop Processor
# ============================================================

def process_video(
    video_output_dir
):

    video_output_dir = (
        Path(
            video_output_dir
        ).resolve()
    )

    video_id = (
        video_output_dir.name
    )


    paths = (
        ensure_phase5_directories(
            video_output_dir
        )
    )


    aligned_dir = (
        paths["aligned_face"]
    )

    landmark_dir = (
        paths["landmarks"]
    )

    metadata_path = (
        paths["alignment_metadata"]
    )

    mouth_dir = (
        paths["mouth_crop"]
    )

    mouth_metadata_path = (
        paths["mouth_metadata"]
    )


    # ========================================================
    # Validate Phase 3
    # ========================================================

    if not aligned_dir.exists():

        raise FileNotFoundError(
            f"Phase 3 aligned_face not found:\n"
            f"{aligned_dir}"
        )


    if not landmark_dir.exists():

        raise FileNotFoundError(
            f"Phase 3 landmarks_aligned not found:\n"
            f"{landmark_dir}"
        )


    # ========================================================
    # Discover
    # ========================================================

    aligned_files = (
        discover_aligned_frames(
            aligned_dir
        )
    )

    landmark_map = (
        build_landmark_map(
            landmark_dir
        )
    )


    if not aligned_files:

        raise RuntimeError(
            f"No aligned frames found:\n"
            f"{aligned_dir}"
        )


    if not landmark_map:

        raise RuntimeError(
            f"No landmark files found:\n"
            f"{landmark_dir}"
        )


    print()
    print("=" * 70)
    print(
        f"PROCESSING : {video_id}"
    )
    print("=" * 70)

    print(
        f"Aligned frames : "
        f"{len(aligned_files)}"
    )

    print(
        f"Landmark files : "
        f"{len(landmark_map)}"
    )


    # ========================================================
    # Existing metadata
    # ========================================================

    metadata_rows = []


    if (
        RESUME_ENABLED
        and
        mouth_metadata_path.exists()
        and
        not FORCE_RERUN
    ):

        try:

            old_df = pd.read_csv(
                mouth_metadata_path
            )

            metadata_rows = (
                old_df
                .to_dict(
                    "records"
                )
            )

        except Exception:

            metadata_rows = []


    existing_frames = {

        int(row["frame_index"])

        for row in metadata_rows

        if (
            "frame_index" in row
            and
            pd.notna(
                row["frame_index"]
            )
        )
    }


    # ========================================================
    # Preview count
    # ========================================================

    preview_count = 0


    # ========================================================
    # Statistics
    # ========================================================

    stats = {

        "video_id":
            video_id,

        "input_frames":
            len(aligned_files),

        "mouth_saved":
            0,

        "skipped":
            0,

        "no_landmarks":
            0,

        "invalid_crop":
            0,

        "errors":
            0,

    }


    # ========================================================
    # Process
    # ========================================================

    for image_path in tqdm(
        aligned_files,
        desc=video_id,
        unit="frame"
    ):

        frame_index = (
            extract_frame_number(
                image_path
            )
        )


        if frame_index is None:

            stats["skipped"] += 1

            continue


        # ----------------------------------------------------
        # Already completed
        # ----------------------------------------------------

        if (
            not FORCE_RERUN
            and
            frame_index
            in existing_frames
        ):

            output_path = (
                mouth_dir /
                image_path.name
            )

            if output_path.exists():

                stats[
                    "mouth_saved"
                ] += 1

                continue


        # ----------------------------------------------------
        # Load image
        # ----------------------------------------------------

        image = cv2.imread(
            str(image_path)
        )


        if image is None:

            stats["errors"] += 1

            metadata_rows.append({

                "video_id":
                    video_id,

                "frame_index":
                    frame_index,

                "image":
                    image_path.name,

                "status":
                    "image_read_error",

            })

            continue


        # ----------------------------------------------------
        # Find landmark
        # ----------------------------------------------------

        landmark_path = (
            landmark_map.get(
                frame_index
            )
        )


        if landmark_path is None:

            stats[
                "no_landmarks"
            ] += 1

            metadata_rows.append({

                "video_id":
                    video_id,

                "frame_index":
                    frame_index,

                "image":
                    image_path.name,

                "landmark_file":
                    None,

                "status":
                    "missing_landmark",

            })

            continue


        # ----------------------------------------------------
        # Load landmark
        # ----------------------------------------------------

        landmarks, landmark_status = (
            load_landmarks(
                landmark_path
            )
        )


        if landmarks is None:

            stats[
                "no_landmarks"
            ] += 1

            metadata_rows.append({

                "video_id":
                    video_id,

                "frame_index":
                    frame_index,

                "image":
                    image_path.name,

                "landmark_file":
                    landmark_path.name,

                "status":
                    landmark_status,

            })

            continue


        # ----------------------------------------------------
        # Calculate ROI
        # ----------------------------------------------------

        try:

            roi_info, roi_status = (
                calculate_mouth_roi(
                    image,
                    landmarks
                )
            )

        except Exception as e:

            stats["errors"] += 1

            metadata_rows.append({

                "video_id":
                    video_id,

                "frame_index":
                    frame_index,

                "image":
                    image_path.name,

                "landmark_file":
                    landmark_path.name,

                "status":
                    f"roi_error:{type(e).__name__}",

            })

            continue


        if roi_info is None:

            stats[
                "invalid_crop"
            ] += 1

            metadata_rows.append({

                "video_id":
                    video_id,

                "frame_index":
                    frame_index,

                "image":
                    image_path.name,

                "landmark_file":
                    landmark_path.name,

                "status":
                    roi_status,

            })

            continue


        crop = (
            roi_info["crop"]
        )


        # ----------------------------------------------------
        # Quality
        # ----------------------------------------------------

        quality = (
            evaluate_crop_quality(
                crop
            )
        )


        # ----------------------------------------------------
        # Save crop
        # ----------------------------------------------------

        output_path = (
            mouth_dir /
            image_path.name
        )


        ok = cv2.imwrite(
            str(output_path),
            crop
        )


        if not ok:

            stats["errors"] += 1

            metadata_rows.append({

                "video_id":
                    video_id,

                "frame_index":
                    frame_index,

                "image":
                    image_path.name,

                "landmark_file":
                    landmark_path.name,

                "status":
                    "write_error",

            })

            continue


        # ----------------------------------------------------
        # Preview
        # ----------------------------------------------------

        if (
            SAVE_PREVIEW
            and
            preview_count
            < PREVIEW_MAX_IMAGES
        ):

            preview_path = (
                paths["mouth_preview"] /
                image_path.name
            )

            cv2.imwrite(
                str(preview_path),
                crop
            )

            preview_count += 1


        # ----------------------------------------------------
        # Metadata
        # ----------------------------------------------------

        metadata_rows.append({

            "video_id":
                video_id,

            "frame_index":
                frame_index,

            "image":
                image_path.name,

            "landmark_file":
                landmark_path.name,

            "center_x":
                roi_info["center_x"],

            "center_y":
                roi_info["center_y"],

            "roi_size":
                roi_info["roi_size"],

            "mouth_width":
                roi_info["mouth_width"],

            "mouth_height":
                roi_info["mouth_height"],

            "x1":
                roi_info["x1"],

            "y1":
                roi_info["y1"],

            "x2":
                roi_info["x2"],

            "y2":
                roi_info["y2"],

            "blur_score":
                quality["blur_score"],

            "brightness":
                quality["brightness"],

            "quality_score":
                quality["score"],

            "quality_status":
                quality["status"],

            "status":
                "saved",

        })


        stats[
            "mouth_saved"
        ] += 1


        # ----------------------------------------------------
        # State
        # ----------------------------------------------------

        write_phase5_state(

            video_output_dir,

            {

                "status":
                    "PROCESSING",

                "next_frame_index":
                    frame_index + 1,

                "video_id":
                    video_id,

                "mouth_saved":
                    stats[
                        "mouth_saved"
                    ],

            }
        )


    # ========================================================
    # Save Metadata
    # ========================================================

    metadata_df = pd.DataFrame(
        metadata_rows
    )


    if not metadata_df.empty:

        metadata_df = (
            metadata_df
            .drop_duplicates(
                subset=[
                    "frame_index"
                ],
                keep="last"
            )
            .sort_values(
                "frame_index"
            )
            .reset_index(
                drop=True
            )
        )


    metadata_df.to_csv(
        mouth_metadata_path,
        index=False,
        encoding="utf-8-sig"
    )


    # ========================================================
    # Summary
    # ========================================================

    summary_df = pd.DataFrame(
        [stats]
    )


    summary_df.to_csv(
        paths["mouth_summary"],
        index=False,
        encoding="utf-8-sig"
    )


    # ========================================================
    # Completed State
    # ========================================================

    write_phase5_state(

        video_output_dir,

        {

            "status":
                "COMPLETED",

            "video_id":
                video_id,

            "input_frames":
                stats[
                    "input_frames"
                ],

            "mouth_saved":
                stats[
                    "mouth_saved"
                ],

            "skipped":
                stats[
                    "skipped"
                ],

            "no_landmarks":
                stats[
                    "no_landmarks"
                ],

            "invalid_crop":
                stats[
                    "invalid_crop"
                ],

            "errors":
                stats[
                    "errors"
                ],

            "completed_at":
                utc_now_iso(),

        }
    )


    return stats


print("=" * 70)
print("Cell 10 — Main Processor Ready")
print("=" * 70)

Cell 10 — Main Processor Ready


In [11]:
# ============================================================
# PHASE 5 — CELL 11
# Discover Video Output Directories
# ============================================================

def discover_video_output_dirs():

    if not OUTPUT_DIR.exists():

        raise FileNotFoundError(
            f"Output directory not found:\n"
            f"{OUTPUT_DIR}"
        )


    dirs = []


    for path in OUTPUT_DIR.iterdir():

        if not path.is_dir():
            continue


        aligned_dir = (
            path /
            ALIGNED_FACE_DIR_NAME
        )


        landmarks_dir = (
            path /
            LANDMARK_DIR_NAME
        )


        if (
            aligned_dir.exists()
            and
            landmarks_dir.exists()
        ):

            dirs.append(
                path.resolve()
            )


    dirs.sort(
        key=lambda p:
            p.name.lower()
    )


    return dirs


VIDEO_OUTPUT_DIRS = (
    discover_video_output_dirs()
)


print("=" * 70)
print("PHASE 5 — VIDEO DISCOVERY")
print("=" * 70)

print(
    f"Videos found: "
    f"{len(VIDEO_OUTPUT_DIRS)}"
)


for index, path in enumerate(
    VIDEO_OUTPUT_DIRS,
    start=1
):

    print(
        f"[{index}] {path.name}"
    )

print("=" * 70)

PHASE 5 — VIDEO DISCOVERY
Videos found: 3
[1] video001
[2] video002
[3] video003


In [12]:
# ============================================================
# PHASE 5 — CELL 12
# Production Batch Runner
# ============================================================

ALL_STATS = []

FAILED_VIDEOS = []


print()
print("=" * 70)
print("PHASE 5 — BATCH PROCESSING")
print("=" * 70)


for index, video_output_dir in enumerate(
    VIDEO_OUTPUT_DIRS,
    start=1
):

    video_id = (
        video_output_dir.name
    )


    print()
    print("=" * 70)

    print(
        f"[{index}/{len(VIDEO_OUTPUT_DIRS)}] "
        f"{video_id}"
    )

    print("=" * 70)


    try:

        stats = process_video(
            video_output_dir
        )


        ALL_STATS.append(
            stats
        )


        if stats["errors"] > 0:

            status = (
                "COMPLETED_WITH_ERRORS"
            )

        else:

            status = (
                "COMPLETED"
            )


        print()
        print(
            f"STATUS       : {status}"
        )

        print(
            f"Input frames : "
            f"{stats['input_frames']}"
        )

        print(
            f"Mouth saved  : "
            f"{stats['mouth_saved']}"
        )

        print(
            f"Skipped      : "
            f"{stats['skipped']}"
        )

        print(
            f"No landmarks : "
            f"{stats['no_landmarks']}"
        )

        print(
            f"Invalid crop : "
            f"{stats['invalid_crop']}"
        )

        print(
            f"Errors       : "
            f"{stats['errors']}"
        )


        if stats["errors"] > 0:

            FAILED_VIDEOS.append(
                {
                    "video_id":
                        video_id,

                    "reason":
                        "processing_errors",

                    "errors":
                        stats["errors"],
                }
            )


    except Exception as e:

        error_text = (
            f"{type(e).__name__}: {e}"
        )


        print()
        print(
            f"FAILED: {video_id}"
        )

        print(
            f"Error: {error_text}"
        )


        traceback.print_exc()


        FAILED_VIDEOS.append(
            {
                "video_id":
                    video_id,

                "reason":
                    error_text,
            }
        )


    finally:

        gc.collect()


print()
print("=" * 70)
print("PHASE 5 — BATCH SUMMARY")
print("=" * 70)


for stats in ALL_STATS:

    status = (
        "COMPLETED_WITH_ERRORS"
        if stats["errors"] > 0
        else
        "COMPLETED"
    )


    print(
        f"{stats['video_id']:<15}"
        f"{status:<25}"
        f"mouth={stats['mouth_saved']:<8}"
        f"skip={stats['skipped']:<8}"
        f"error={stats['errors']}"
    )


if FAILED_VIDEOS:

    print()
    print("-" * 70)
    print("VIDEOS REQUIRING ATTENTION")
    print("-" * 70)


    for item in FAILED_VIDEOS:

        print(
            f"{item['video_id']:<15}"
            f"{item['reason']}"
        )

else:

    print()
    print(
        "All videos processed successfully."
    )


print("=" * 70)


PHASE 5 — BATCH PROCESSING

[1/3] video001

PROCESSING : video001
Aligned frames : 41058
Landmark files : 44437


video001:   0%|          | 0/41058 [00:00<?, ?frame/s]


STATUS       : COMPLETED
Input frames : 41058
Mouth saved  : 40746
Skipped      : 0
No landmarks : 312
Invalid crop : 0
Errors       : 0

[2/3] video002

PROCESSING : video002
Aligned frames : 34794
Landmark files : 53581


video002:   0%|          | 0/34794 [00:00<?, ?frame/s]


STATUS       : COMPLETED
Input frames : 34794
Mouth saved  : 34278
Skipped      : 0
No landmarks : 516
Invalid crop : 0
Errors       : 0

[3/3] video003

PROCESSING : video003
Aligned frames : 28276
Landmark files : 28276


video003:   0%|          | 0/28276 [00:00<?, ?frame/s]


STATUS       : COMPLETED
Input frames : 28276
Mouth saved  : 28110
Skipped      : 0
No landmarks : 166
Invalid crop : 0
Errors       : 0

PHASE 5 — BATCH SUMMARY
video001       COMPLETED                mouth=40746   skip=0       error=0
video002       COMPLETED                mouth=34278   skip=0       error=0
video003       COMPLETED                mouth=28110   skip=0       error=0

All videos processed successfully.


In [6]:
# ============================================================
# PHASE 5 — CELL 13
# Final Validation
# ============================================================

from pathlib import Path
import pandas as pd
import re


# ============================================================
# Discover frame IDs
# ============================================================

def discover_frame_ids(frame_dir):
    """
    Discover frame IDs from files inside a directory.

    รองรับชื่อประมาณ:
        frame_000000.jpg
        frame_000001.png
        000000.jpg
        landmark_000000.json
        frame_000000_landmarks.json

    ใช้เลขชุดสุดท้ายในชื่อไฟล์เป็น frame ID
    """

    frame_dir = Path(frame_dir).resolve()

    if not frame_dir.exists():
        return set()

    frame_ids = set()

    # ใช้ rglob เพื่อรองรับ subdirectories
    for p in frame_dir.rglob("*"):

        if not p.is_file():
            continue

        name = p.stem.lower()

        numbers = re.findall(r"\d+", name)

        if not numbers:
            continue

        try:
            frame_id = int(numbers[-1])
            frame_ids.add(frame_id)
        except ValueError:
            continue

    return frame_ids


# ============================================================
# Get Phase 5 paths
# ============================================================

def get_phase5_paths(video_output_dir):

    video_output_dir = Path(video_output_dir).resolve()

    return {
        "aligned_face":
            video_output_dir / "aligned_face",

        "landmarks":
            video_output_dir / "landmarks",

        "alignment_metadata":
            video_output_dir / "alignment_metadata.csv",

        "mouth_metadata":
            video_output_dir / "mouth_metadata.csv",
    }


# ============================================================
# Read frame index column safely
# ============================================================

def read_frame_indices(df, column="frame_index"):

    if df.empty:
        return set()

    if column not in df.columns:
        return set()

    values = pd.to_numeric(
        df[column],
        errors="coerce"
    )

    values = values.dropna().astype(int)

    return set(values)


# ============================================================
# Validate one video
# ============================================================

def validate_phase5_video(video_output_dir):

    video_output_dir = Path(
        video_output_dir
    ).resolve()

    video_id = video_output_dir.name

    paths = get_phase5_paths(
        video_output_dir
    )


    # ========================================================
    # Aligned frames
    # ========================================================

    aligned_frames = discover_frame_ids(
        paths["aligned_face"]
    )


    # ========================================================
    # Landmark frames
    # ========================================================

    landmark_frames = discover_frame_ids(
        paths["landmarks"]
    )


    # ========================================================
    # Alignment metadata
    # ========================================================

    if paths["alignment_metadata"].exists():

        try:

            alignment_df = pd.read_csv(
                paths["alignment_metadata"],
                low_memory=False
            )

        except Exception as e:

            print(
                f"⚠️ Cannot read alignment metadata: {e}"
            )

            alignment_df = pd.DataFrame()

    else:

        alignment_df = pd.DataFrame()


    metadata_frames = read_frame_indices(
        alignment_df,
        "frame_index"
    )


    # ========================================================
    # Mouth metadata
    # ========================================================

    if paths["mouth_metadata"].exists():

        try:

            mouth_df = pd.read_csv(
                paths["mouth_metadata"],
                low_memory=False
            )

        except Exception as e:

            print(
                f"⚠️ Cannot read mouth metadata: {e}"
            )

            mouth_df = pd.DataFrame()

    else:

        mouth_df = pd.DataFrame()


    mouth_ok = set()

    insufficient_landmarks = set()

    mouth_metadata_ids = set()

    unknown_status = 0


    if not mouth_df.empty:

        if "frame_index" in mouth_df.columns:

            mouth_df["frame_index"] = pd.to_numeric(
                mouth_df["frame_index"],
                errors="coerce"
            )

            mouth_df = mouth_df.dropna(
                subset=["frame_index"]
            )

            mouth_df["frame_index"] = (
                mouth_df["frame_index"]
                .astype(int)
            )

            mouth_metadata_ids = set(
                mouth_df["frame_index"]
            )


        if "status" in mouth_df.columns:

            mouth_ok = set(
                mouth_df[
                    mouth_df["status"] == "OK"
                ]["frame_index"]
            )

            insufficient_landmarks = set(
                mouth_df[
                    mouth_df["status"]
                    == "insufficient_landmarks"
                ]["frame_index"]
            )


            # ------------------------------------------------
            # Unknown status
            # ------------------------------------------------

            valid_status = {
                "OK",
                "insufficient_landmarks"
            }

            unknown_status = int(
                (
                    ~mouth_df["status"].isin(
                        valid_status
                    )
                ).sum()
            )


    # ========================================================
    # Expected mouth OK
    # ========================================================

    expected_mouth_ok = (
        metadata_frames
        & aligned_frames
        - insufficient_landmarks
    )


    # ========================================================
    # Relationships
    # ========================================================

    aligned_no_landmark = (
        aligned_frames
        - landmark_frames
    )

    landmark_no_aligned = (
        landmark_frames
        - aligned_frames
    )

    metadata_no_aligned = (
        metadata_frames
        - aligned_frames
    )

    mouth_no_metadata = (
        mouth_ok
        - metadata_frames
    )

    mouth_no_aligned = (
        mouth_ok
        - aligned_frames
    )

    insufficient_no_metadata = (
        insufficient_landmarks
        - metadata_frames
    )

    insufficient_no_aligned = (
        insufficient_landmarks
        - aligned_frames
    )


    # ========================================================
    # Duplicates
    # ========================================================

    alignment_duplicates = 0

    if (
        not alignment_df.empty
        and "frame_index" in alignment_df.columns
    ):

        alignment_duplicates = int(
            alignment_df["frame_index"]
            .duplicated()
            .sum()
        )


    mouth_duplicates = 0

    if (
        not mouth_df.empty
        and "frame_index" in mouth_df.columns
    ):

        mouth_duplicates = int(
            mouth_df["frame_index"]
            .duplicated()
            .sum()
        )


    # ========================================================
    # ERRORS
    #
    # สิ่งเหล่านี้กระทบความถูกต้องของ Phase 5
    # ========================================================

    errors = []


    if aligned_no_landmark:

        errors.append(
            "aligned_without_landmark"
        )


    if metadata_no_aligned:

        errors.append(
            "metadata_without_aligned"
        )


    if mouth_no_metadata:

        errors.append(
            "mouth_without_metadata"
        )


    if mouth_no_aligned:

        errors.append(
            "mouth_without_aligned"
        )


    if insufficient_no_metadata:

        errors.append(
            "insufficient_without_metadata"
        )


    if insufficient_no_aligned:

        errors.append(
            "insufficient_without_aligned"
        )


    if unknown_status:

        errors.append(
            "unknown_status"
        )


    if alignment_duplicates:

        errors.append(
            "alignment_duplicates"
        )


    if mouth_duplicates:

        errors.append(
            "mouth_duplicates"
        )


    # ========================================================
    # WARNINGS
    #
    # Landmark ที่ไม่มี aligned ไม่ทำให้ Phase 5 ตก
    #
    # เพราะ Phase 5 ใช้ aligned frames เป็น input หลัก
    # ========================================================

    warnings = []


    if landmark_no_aligned:

        warnings.append(
            "landmark_without_aligned"
        )


    # ========================================================
    # Final OK
    #
    # ต้องตรวจเฉพาะ consistency ที่เกี่ยวข้องกับ
    # mouth crop
    # ========================================================

    final_ok = (
        len(errors) == 0
        and len(metadata_frames) > 0
        and len(mouth_metadata_ids) > 0
        and len(mouth_ok)
        == len(expected_mouth_ok)
    )


    # ========================================================
    # Result
    # ========================================================

    result = {

        "video_id":
            video_id,

        "aligned_frames":
            len(aligned_frames),

        "landmark_frames":
            len(landmark_frames),

        "metadata_frames":
            len(metadata_frames),

        "mouth_ok":
            len(mouth_ok),

        "insufficient_landmarks":
            len(insufficient_landmarks),

        "expected_mouth_ok":
            len(expected_mouth_ok),

        "mouth_metadata_ids":
            len(mouth_metadata_ids),

        "aligned_no_landmark":
            len(aligned_no_landmark),

        "landmark_no_aligned":
            len(landmark_no_aligned),

        "metadata_no_aligned":
            len(metadata_no_aligned),

        "mouth_no_metadata":
            len(mouth_no_metadata),

        "mouth_no_aligned":
            len(mouth_no_aligned),

        "insufficient_no_metadata":
            len(insufficient_no_metadata),

        "insufficient_no_aligned":
            len(insufficient_no_aligned),

        "unknown_status":
            unknown_status,

        "alignment_duplicates":
            alignment_duplicates,

        "mouth_duplicates":
            mouth_duplicates,

        "warnings":
            ";".join(warnings),

        "errors":
            ";".join(errors),

        "final_ok":
            final_ok,
    }


    # ========================================================
    # Print
    # ========================================================

    print()

    print("=" * 70)
    print(
        f"VIDEO: {video_id}"
    )
    print("=" * 70)

    print(
        f"Aligned frames          : "
        f"{len(aligned_frames)}"
    )

    print(
        f"Landmark frames         : "
        f"{len(landmark_frames)}"
    )

    print(
        f"Metadata frames         : "
        f"{len(metadata_frames)}"
    )

    print(
        f"Mouth OK                : "
        f"{len(mouth_ok)}"
    )

    print(
        f"Insufficient landmarks  : "
        f"{len(insufficient_landmarks)}"
    )

    print(
        f"Expected mouth OK       : "
        f"{len(expected_mouth_ok)}"
    )

    print(
        f"Aligned → no landmark   : "
        f"{len(aligned_no_landmark)}"
    )

    print(
        f"Landmark → no aligned   : "
        f"{len(landmark_no_aligned)}"
    )

    print(
        f"Metadata → no aligned   : "
        f"{len(metadata_no_aligned)}"
    )

    print(
        f"Mouth OK → no metadata  : "
        f"{len(mouth_no_metadata)}"
    )

    print(
        f"Mouth OK → no aligned   : "
        f"{len(mouth_no_aligned)}"
    )

    print(
        f"Insufficient → no meta  : "
        f"{len(insufficient_no_metadata)}"
    )

    print(
        f"Insufficient → no align : "
        f"{len(insufficient_no_aligned)}"
    )

    print(
        f"Unknown status          : "
        f"{unknown_status}"
    )

    print(
        f"Alignment duplicates    : "
        f"{alignment_duplicates}"
    )

    print(
        f"Mouth duplicates        : "
        f"{mouth_duplicates}"
    )


    # ========================================================
    # Warnings
    # ========================================================

    if warnings:

        print()
        print("Validation warnings:")

        for warning in warnings:

            if warning == "landmark_without_aligned":

                print(
                    f"  ⚠️ {warning} "
                    f"({len(landmark_no_aligned)} frames)"
                )

            else:

                print(
                    f"  ⚠️ {warning}"
                )


    # ========================================================
    # Errors
    # ========================================================

    if errors:

        print()
        print("Validation errors:")

        for error in errors:

            print(
                f"  ❌ {error}"
            )


    else:

        print()
        print(
            "✅ No validation errors"
        )


    # ========================================================
    # Final status
    # ========================================================

    if final_ok:

        print()
        print(
            "✅ PHASE 5 VIDEO VALIDATION PASSED"
        )

    else:

        print()
        print(
            "❌ PHASE 5 VIDEO VALIDATION FAILED"
        )


    return result


# ============================================================
# PHASE 5 — VALIDATION INPUT
# ============================================================

VIDEO_OUTPUT_DIRS = sorted(
    [
        p
        for p in OUTPUT_ROOT.iterdir()
        if (
            p.is_dir()
            and p.name.lower().startswith("video")
        )
    ],
    key=lambda p: p.name.lower()
)


print("=" * 70)
print("PHASE 5 — VALIDATION INPUT")
print("=" * 70)

for video_dir in VIDEO_OUTPUT_DIRS:

    print(
        f"{video_dir.name}: "
        f"{video_dir.exists()}"
    )


print("=" * 70)
print(
    "PHASE 5 — VALIDATION INPUT"
)
print("=" * 70)

for video_dir in VIDEO_OUTPUT_DIRS:

    print(
        f"{video_dir.name}: "
        f"{video_dir.exists()}"
    )


# ============================================================
# Run validation
# ============================================================

FINAL_ROWS = []

for video_output_dir in VIDEO_OUTPUT_DIRS:

    result = validate_phase5_video(
        video_output_dir
    )

    FINAL_ROWS.append(
        result
    )


# ============================================================
# Final DataFrame
# ============================================================

FINAL_VALIDATION_DF = pd.DataFrame(
    FINAL_ROWS
)


print()

print("=" * 70)
print(
    "PHASE 5 — FINAL VALIDATION"
)
print("=" * 70)

if not FINAL_VALIDATION_DF.empty:

    print(
        FINAL_VALIDATION_DF.to_string(
            index=False
        )
    )


print("=" * 70)

if (
    not FINAL_VALIDATION_DF.empty
    and FINAL_VALIDATION_DF["final_ok"].all()
):

    print(
        "✅ PHASE 5 VALIDATION PASSED"
    )

else:

    print(
        "⚠️ PHASE 5 VALIDATION REQUIRES ATTENTION"
    )

print("=" * 70)

PHASE 5 — VALIDATION INPUT
video001: True
video002: True
video003: True
PHASE 5 — VALIDATION INPUT
video001: True
video002: True
video003: True

VIDEO: video001
Aligned frames          : 41058
Landmark frames         : 44437
Metadata frames         : 41058
Mouth OK                : 40746
Insufficient landmarks  : 312
Expected mouth OK       : 40746
Aligned → no landmark   : 0
Landmark → no aligned   : 3379
Metadata → no aligned   : 0
Mouth OK → no metadata  : 0
Mouth OK → no aligned   : 0
Insufficient → no meta  : 0
Insufficient → no align : 0
Unknown status          : 0
Alignment duplicates    : 0
Mouth duplicates        : 0

Validation warnings:
  ⚠️ landmark_without_aligned (3379 frames)

✅ No validation errors

✅ PHASE 5 VIDEO VALIDATION PASSED

VIDEO: video002
Aligned frames          : 34794
Landmark frames         : 53581
Metadata frames         : 34794
Mouth OK                : 34278
Insufficient landmarks  : 516
Expected mouth OK       : 34278
Aligned → no landmark   : 0
Landma